In [2]:
import pandas as pd

In [3]:
movie_des = pd.read_csv("../data/enriched_movie_descriptions.csv")

In [5]:
movie_original = pd.read_csv("../data/movie.csv")
movie_rating= pd.read_csv("../data/rating.csv")

In [26]:
movie_rating.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [6]:
movie_des.head()

,movieId,tmdbId,title,description
0,1,862,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,2,8844,Jumanji,When siblings Judy and Peter discover an encha...
2,3,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,4,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,5,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...


In [8]:
movie_des.count()

movieId        26634
tmdbId         26634
title          26634
description    26619
dtype: int64

In [9]:
movie_original.count()

movieId    27278
title      27278
genres     27278
dtype: int64

In [10]:
import pandas as pd

# Step 1: Remove rows in 'movie_des' where the 'description' is missing
# This leaves us with the 26,619 movies that actually have a description.
movie_des_filtered = movie_des.dropna(subset=['description'])

# Step 2: Combine 'movie_original' with the filtered descriptions
# We only take 'movieId' and 'description' from the filtered dataset 
# to avoid getting duplicate columns for 'title' (like 'title_x' and 'title_y').
process_movie = pd.merge(
    movie_original, 
    movie_des_filtered[['movieId', 'description']], 
    on='movieId', 
    how='inner' 
)

# Step 3: Rearrange columns to match your exact requested format (optional but clean)
process_movie = process_movie[['movieId', 'title', 'genres', 'description']]

# View the results
print(process_movie.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26619 entries, 0 to 26618
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   movieId      26619 non-null  int64 
 1   title        26619 non-null  object
 2   genres       26619 non-null  object
 3   description  26619 non-null  object
dtypes: int64(1), object(3)
memory usage: 832.0+ KB
None


In [11]:
process_movie.head()

,movieId,title,genres,description
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"Led by Woody, Andy's toys live happily in his ..."
1,2,Jumanji (1995),Adventure|Children|Fantasy,When siblings Judy and Peter discover an encha...
2,3,Grumpier Old Men (1995),Comedy|Romance,A family wedding reignites the ancient feud be...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,"Cheated on, mistreated and stepped on, the wom..."
4,5,Father of the Bride Part II (1995),Comedy,Just when George Banks has recovered from his ...


In [13]:
process_movie['year'] = process_movie['title'].str.extract(r'\((\d{4})\)')

In [15]:
# Remove the space and the year in parentheses from the end of the title
# r'\s*\(\d{4}\)$' looks for: a space, then (four digits), at the end of the text
process_movie['title'] = process_movie['title'].str.replace(r'\s*\(\d{4}\)$', '', regex=True)

In [24]:
# Replace the pipe '|' with a comma and a space ', '
process_movie['genres'] = process_movie['genres'].str.replace('|', ',', regex=False)

In [25]:
process_movie.head()

,movieId,title,genres,description,year
0,1,Toy Story,"Adventure,Animation,Children,Comedy,Fantasy","Led by Woody, Andy's toys live happily in his ...",1995
1,2,Jumanji,"Adventure,Children,Fantasy",When siblings Judy and Peter discover an encha...,1995
2,3,Grumpier Old Men,"Comedy,Romance",A family wedding reignites the ancient feud be...,1995
3,4,Waiting to Exhale,"Comedy,Drama,Romance","Cheated on, mistreated and stepped on, the wom...",1995
4,5,Father of the Bride Part II,Comedy,Just when George Banks has recovered from his ...,1995


In [27]:
# Step 4: Filter movie_rating to keep only IDs that exist in process_movie
movie_rating = movie_rating[movie_rating['movieId'].isin(process_movie['movieId'])]

# Optional: Check how many rows are left
print(f"Ratings remaining: {len(movie_rating)}")
print(movie_rating.head())

Ratings remaining: 19949440
   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40


In [30]:
# Save the DataFrame to a CSV file in your current working directory
process_movie.to_csv('process_movie.csv', index=False)

In [29]:
movie_rating.to_csv('process_movie_rating.csv', index=False)